# UC2 — HEK293 heterogeneous biology as one AnnNet object

This use case builds a cell-line-specific map of human cell biology as one AnnNet
object. It then queries, validates, exports, and trains a model on that object.
It is a software case study and reports no biological finding.

The graph fuses four sources: OmniPath protein–protein interactions, OmniPath
complexes, the Human-GEM metabolic model, and DoRothEA regulation. AnnNet holds
all four in one container. The whole series spans about 600 lines of notebook
code; the same graph across a separate graph library, a hypergraph library, and a
metabolic-model toolkit requires far more code and manual synchronisation.

## The multilayer model

AnnNet applies the Kivelä formalism, which layers a network along independent
aspects. This use case uses two aspects.

- `mechanism` ∈ {signaling, metabolic, regulatory}: the process a molecule
  participates in.
- `complex` ∈ {member, monomer}: whether a protein belongs to a complex.

The two aspects are orthogonal. A protein signals while it belongs to a complex.
Each protein is therefore one node-layer, for example `(prot:TP53, signaling,
member)`. The two-aspect design adds no identity-coupling edges.

## How to run

Run the notebooks in order, `01` through `07`. Notebook `02` builds the graph and
writes a snapshot. Each later notebook reloads that snapshot with
`uc2_common.load()`. Shared constants and helpers live in `uc2_common.py`, so each
cell shows the operation rather than the boilerplate.

## Part 1 — Cell-line gate

Every downstream source intersects one protein set: the genes the Human Protein
Atlas records as expressed in HEK293 with protein-level evidence. The cell below
applies this filter once and caches the result, with each protein's compartment,
for `02_build`. The filter retains 18,557 proteins.

In [1]:
import zipfile

import polars as pl
import requests

from uc2_common import DATA, HEK293, compartment_code

tsv = DATA / "proteinatlas.tsv"
if not tsv.exists():
    z = DATA / "proteinatlas.tsv.zip"
    z.write_bytes(requests.get("https://www.proteinatlas.org/download/proteinatlas.tsv.zip", timeout=300).content)
    zipfile.ZipFile(z).extractall(DATA)

hpa = pl.read_csv(tsv, separator="\t", infer_schema_length=10000)
expressed = (
    hpa.filter(pl.col("Evidence").str.contains("protein level")
               | pl.col("UniProt evidence").str.contains("protein level"))
    .select(Gene="Gene", Uniprot="Uniprot", Ensembl="Ensembl",
            hpa_location="Subcellular main location")
    .with_columns(compartment=pl.col("hpa_location").map_elements(compartment_code, return_dtype=pl.Utf8))
)
expressed.write_parquet(HEK293)
print(f"HEK293 expressed proteins (protein-level evidence): {expressed.height:,}")
print(expressed.head(3))

HEK293 expressed proteins (protein-level evidence): 18,564
shape: (3, 5)
┌────────┬─────────┬─────────────────┬─────────────────────────┬─────────────┐
│ Gene   ┆ Uniprot ┆ Ensembl         ┆ hpa_location            ┆ compartment │
│ ---    ┆ ---     ┆ ---             ┆ ---                     ┆ ---         │
│ str    ┆ str     ┆ str             ┆ str                     ┆ str         │
╞════════╪═════════╪═════════════════╪═════════════════════════╪═════════════╡
│ TSPAN6 ┆ O43657  ┆ ENSG00000000003 ┆ Cell Junctions, Cytosol ┆ c           │
│ TNMD   ┆ Q9H2S6  ┆ ENSG00000000005 ┆ null                    ┆ null        │
│ DPM1   ┆ O60762  ┆ ENSG00000000419 ┆ null                    ┆ null        │
└────────┴─────────┴─────────────────┴─────────────────────────┴─────────────┘
